# Joint Disambiguation Model Demo

This notebook acts as a quick tour of the attention-based model that improves Gilda's entity grounding by disambiguating all of a document's mentions jointly. The cells below illustrate what the model consists of, how it works, and how it compares to plain Gilda on individual examples and a held-out corpus.

## 1. The problem

Grounding is the process of taking a text mention (e.g. `"ER"`) and assigning it a specific database entity (e.g. `HGNC:3467`). Gilda does this by lexical string matching, returning a ranked list of candidates per mention. Mentions can often be ambiguous, with one mention form being used in different settings to refer to different entities (e.g., `"ER"` can refer to `"estrogen receptor"` or `"endoplasmic reticulum"`). Gilda works around this by optionally consuming the text surrounding a mention, but cannot for input data types that lack this additional text (e.g., a table of mentions). For these data types, the only context available to help resolve ambiguous mentions is other mentions from the same source; humans often disambiguate this way: for instance, a well-trained scientist who sees the mentions `"breast cancer"`, `"tamoxifen"`, and `"ER"` can infer that `"ER"` means `"estrogen receptor"` in this setting based on the other two and their background knowledge of breast cancer. The model showcased below attempts to mimic this human-like disambiguation strategy with two key tools: (1) a frozen PubMedBERT model to encode the scientific meaning of text spans, and (2) an attention mechanism trained to use those encodings to resolve ambiguous mentions. More concretely, the model re-ranks the gilda-generated candidate lists of all mentions in a document jointly by letting related mentions inform each other.

## 2. How the model works

For each document...
1. Every mention is generated a candidate list with Gilda
2. Every candidate for every mention becomes a token and given an embedding by a frozen PubMedBERT encoder, plus Gilda's score "ingredients" (its status + string-match sub-scores) concatenated to the end to make the full representation.
3. All candidates in the document are run through self-attention, allowing candidates from different mentions to influence each other as they update their representations.
4. A score head reads each candidate's post-attention representation with its raw Gilda sub-score block re-appended (a **feature skip**, so lexical evidence reaches the head without passing through a LayerNorm) and scores the candidate directly — there is no gate blending the model's score back into Gilda's.

The model loaded here (`us_lr5e5_vrex_ns`) adds two variations: a small **attention bias** determined by each candidates `status` and `exact` Gilda sub-score values (to allow the model to attend more to trusted, exact-match candidates), and the addition of a **document-context token**, whose embedding is derived from passing a comma-separated string of all the document's mentions through PubMedBERT, serving like a summary of document sentiment that candidates can attend to. It is also trained with two extras: an **auxiliary entity-type head** (predicting a mention's semantic type alongside the ranking loss) and a **V-REx penalty** that pushes the training loss to be equally low across documents grouped by their gold-namespace mix, discouraging the model from leaning on namespace priors.

## 3. Setup

Loads the pre-built corpus and caches straight from disk (no Gilda grounder or BERT needed), then loads the trained model. Runs in a few seconds.

In [31]:
import os
import pickle
import pandas as pd

if os.path.basename(os.getcwd()) == "ascenda":
    os.chdir("..")

from ascenda.data import load_corpus, make_splits
from ascenda.train import load_model, predict_document
from ascenda.evaluate import filter_ambiguous_mentions, comparison_table

datasets = ["bioid", "bc5cdr", "nlmchem", "ncbi_disease", "gnormplus", "medmentions_st21pv"]

docs = load_corpus(datasets, merged_cache="ascenda/caches/corpus_cache/final_merged.pkl")
_, _, test = make_splits(docs)
emb_cache = pickle.load(open("ascenda/caches/embedding_cache_final.pkl", "rb"))
ctx_cache = pickle.load(open("ascenda/caches/context_cache_final.pkl", "rb"))

model = load_model("ascenda/models/final_model_best_checkpoint_seed0.pt")
print(f"Loaded {os.path.basename("ascenda/models/final_model_best_checkpoint_seed0.pt")}  |  {len(test)} test documents")

Loaded merged corpus (8067 docs) from ascenda/caches/corpus_cache/final_merged.pkl
Loaded final_model_best_checkpoint_seed0.pt  |  1868 test documents


## 4. Gilda vs. Model on a single document

For a single document, view each mention's **Gilda** pick next to what the **model** picks, and whether those picks are correct. The model "fixes" a mention's grounding when its pick is correct and Gilda's is not. It does so by using **other mentions** in the document as context (and importantly, not the document's raw text).

In [32]:
def curie(c):
    return f"{c.term.db}:{c.term.id}"

def compare_doc(doc):
    preds = predict_document(doc, emb_cache, model, context_cache=ctx_cache)
    rows = []
    for m in doc.mentions:
        if not m.candidates:
            continue
        gilda_top = m.candidates[0]
        model_top = preds.get(m.text, m.candidates)[0]
        rows.append({
            "mention": m.text,
            "type": m.entity_type,
            "Gilda pick": f"{gilda_top.term.entry_name} ({curie(gilda_top)})",
            "Model pick": f"{model_top.term.entry_name} ({curie(model_top)})",
            "Gilda Correct": (curie(gilda_top) in m.gold_synonyms),
            "Model Correct": (curie(model_top) in m.gold_synonyms),
        })
    return pd.DataFrame(rows)

ambiguous = filter_ambiguous_mentions(test)

def model_gains(doc):
    df = compare_doc(doc)
    return len(df) and (~df["Gilda Correct"] & df["Model Correct"]).any()

examples = [d for d in ambiguous if 2 <= len(d.mentions) <= 6 and model_gains(d)]
print("Document:", examples[10].doc_id)
compare_doc(examples[10])

Ambiguity filter: kept 6020/68366 mentions (8.8%) across 1004 docs (gap<=0.05, candidates>=2, top>=0.3)
Document: PMC:4403043


,mention,type,Gilda pick,Model pick,Gilda Correct,Model Correct
0,TA,Tissue/Organ,Takayasu Arteritis (MESH:D013625),Eda (UP:O54693),False,False
1,dystrophin,Nonhuman Gene,DMD (HGNC:2928),Dmd (UP:P11531),False,True


## 5. Gilda vs. Model on a real dataset

Here we get a full picture by re-ranking every mention in a corpus of held-out documents with the model and comparing against Gilda, breaking down performance by entity type. The first cell demonstrates this on a full held-out set of documents, and the second on a filtered subset of especially ambiguous mentions from the same corpus.

Comparison table column definitions:

- *Entity Type* – semantic type of the gold entity that a mention refers to
- *Total* – number of mentions across all documents in the corpus
- *Gilda F1* – F1 score attained by grounding with Gilda (no context disambiguation)
- *Model F1* – F1 score attained by re-ranking Gilda candidates with the joint disambiguation model (no context disambiguation)
- *Gains* – Number of previously-incorrect groundings corrected by the joint disambiguation model
- *Losses* – Number of previously-correct groundings made incorrect by the joint disambiguation model
- *Net* – *Gains* minus *Losses*
- *Net %* – *Net* divided by *Total* times 100
- *G/L* – *Gains* divided by *Losses*

The `All` row is the main result: **Model F1** should be greater than **Gilda F1** and **Net** should be greater than 0 if the model corrects more than it breaks. Per type, it helps most where document context is informative (e.g. Disease, Biological Function, Taxon) and is at least better than neutral on others (G/L > 1.0).

In [33]:
print("Full test corpus")
preds = {d.doc_id: predict_document(d, emb_cache, model, context_cache=ctx_cache)
         for d in test}
comparison_table(test, preds, model_col="Model F1", total_label="All", add_average_row=False)

Full test corpus


,Entity Type,Total,Gilda F1,Model F1,Gains,Losses,Net,G/L,Net %
0,Biological Function,8113,0.489,0.537,316,12,304,26.33,3.7
1,Cell types/Cell lines,348,0.605,0.612,2,0,2,inf,0.6
2,Cellular Component,356,0.493,0.546,16,0,16,inf,4.5
3,Disease,8976,0.482,0.537,401,3,398,133.67,4.4
4,Human Gene,3384,0.813,0.841,141,52,89,2.71,2.6
5,Nonhuman Gene,859,0.463,0.494,26,1,25,26.00,2.9
6,Small Molecule,24764,0.404,0.468,1510,67,1443,22.54,5.8
7,Taxon,2580,0.531,0.568,80,8,72,10.00,2.8
8,Tissue/Organ,4377,0.527,0.534,24,0,24,inf,0.5
9,other,14609,0.289,0.293,44,0,44,inf,0.3


In [35]:
print("Ambiguous mentions only")
ambig_preds = {d.doc_id: predict_document(d, emb_cache, model, context_cache=ctx_cache)
         for d in ambiguous}
comparison_table(ambiguous, ambig_preds, model_col="Model F1", total_label="All", add_average_row=False)

Ambiguous mentions only


,Entity Type,Total,Gilda F1,Model F1,Gains,Losses,Net,G/L,Net %
0,Biological Function,783,0.386,0.670,229,6,223,38.17,28.5
1,Cell types/Cell lines,28,0.000,0.071,2,0,2,inf,7.1
2,Cellular Component,39,0.667,0.795,5,0,5,inf,12.8
3,Disease,739,0.235,0.686,333,0,333,inf,45.1
4,Human Gene,302,0.563,0.616,70,54,16,1.30,5.3
5,Nonhuman Gene,85,0.212,0.341,12,1,11,12.00,12.9
6,Small Molecule,3499,0.245,0.448,750,43,707,17.44,20.2
7,Taxon,120,0.033,0.608,69,0,69,inf,57.5
8,Tissue/Organ,138,0.326,0.355,4,0,4,inf,2.9
9,other,287,0.063,0.178,33,0,33,inf,11.5


## 6. Examples

The cell below mines the held-out corpus in search of example documents where the model fixes Gilda mistakes. Run it with different values of n to see different output sizes.

In [38]:
use_ambiguous = True  # flip to False if you want to look at example docs from the full test set

if use_ambiguous:
    by_id = {d.doc_id: d for d in ambiguous}
else:
    by_id = {d.doc_id: d for d in test}

def doc_flips(doc):
    preds = predict_document(doc, emb_cache, model, context_cache=ctx_cache)
    gains = []
    losses = []
    seen = set()
    for m in doc.mentions:
        if not m.candidates or m.text in seen:
            continue
        seen.add(m.text)
        g = m.candidates[0]
        mo = preds.get(m.text, m.candidates)[0]
        g_ok, mo_ok = curie(g) in m.gold_synonyms, curie(mo) in m.gold_synonyms
        if mo_ok and not g_ok:
            gains.append((m.text, g, mo))
        elif g_ok and not mo_ok:
            losses.append((m.text, g, mo))
    return gains, losses

def mine_examples(docs, n=10, max_mentions=8):
    scored = []
    for d in docs:
        if not (1 <= len(d.mentions) <= max_mentions):
            continue
        gains, losses = doc_flips(d)
        if not gains:
            continue
        cross = sum(g.term.db != mo.term.db for _, g, mo in gains)
        scored.append(((len(losses) == 0), cross, len(gains), d.doc_id, gains))
    scored.sort(key=lambda t: (t[0], t[1], t[2]), reverse=True)
    return {
        doc_id: "".join(
            f"  Fixed {t} -> {mo.term.entry_name} ({mo.term.db})"
            f"  [Gilda choice: {g.term.entry_name} ({g.term.db})]\n"
            for t, g, mo in gains[:3])
        for _, _, _, doc_id, gains in scored[:n]
    }

mined_examples = mine_examples(ambiguous)

for doc_id, note in mined_examples.items():
    print(f"\n{doc_id}: \n{note}")
    df = compare_doc(by_id[doc_id])
    display(df.drop_duplicates(subset=["mention", "Model pick"]))


PMC:4801944: 
  Fixed TNF-α -> Tnf (UP)  [Gilda choice: TNF (HGNC)]
  Fixed DSS -> dextran sulfate (CHEBI)  [Gilda choice: NR0B1 (HGNC)]
  Fixed IL-23p19 -> Il23a (UP)  [Gilda choice: IL23A (HGNC)]



,mention,type,Gilda pick,Model pick,Gilda Correct,Model Correct
0,TNF-α,Nonhuman Gene,TNF (HGNC:11892),Tnf (UP:P06804),False,True
1,DSS,Small Molecule,NR0B1 (HGNC:7960),dextran sulfate (CHEBI:CHEBI:34674),False,True
2,IL-23p19,Nonhuman Gene,IL23A (HGNC:15488),Il23a (UP:Q9EQ14),False,True



PMC:5278607: 
  Fixed ER -> endoplasmic reticulum (GO)  [Gilda choice: ESR (FPLX)]
  Fixed PM -> plasma membrane (GO)  [Gilda choice: PRB1 (HGNC)]
  Fixed AA -> antimycin A (CHEBI)  [Gilda choice: SAA (FPLX)]



,mention,type,Gilda pick,Model pick,Gilda Correct,Model Correct
0,cER,Cellular Component,btuB (UP:P06129),ceramide (CHEBI:CHEBI:17761),False,False
1,ER,Cellular Component,ESR (FPLX:ESR),endoplasmic reticulum (GO:GO:0005783),False,True
2,PDI,Human Gene,PDIA2 (HGNC:14180),PADI1 (HGNC:18367),False,False
3,PM,Cellular Component,PRB1 (HGNC:9337),plasma membrane (GO:GO:0005886),False,True
4,OM,Small Molecule,OCM (HGNC:8105),OCM (HGNC:8105),False,False
5,AA,Small Molecule,SAA (FPLX:SAA),antimycin A (CHEBI:CHEBI:22584),False,True



PMID:12535818: 
  Fixed AF -> Atrial Fibrillation (MESH)  [Gilda choice: Actin (FPLX)]
  Fixed MI -> Myocardial Infarction (MESH)  [Gilda choice: MITF (HGNC)]
  Fixed digoxin -> Digoxin (MESH)  [Gilda choice: digoxin (CHEBI)]



...

,mention,type,Gilda pick,Model pick,Gilda Correct,Model Correct
0,PD-1,Nonhuman Gene,protectin D1 (CHEBI:CHEBI:138655),P4hb (UP:P09103),False,False
1,IgG,Cellular Component,IgG immunoglobulin complex (GO:GO:0071735),IgG immunoglobulin complex (GO:GO:0071735),True,True
2,2APB,Small Molecule,2-aminoethoxydiphenyl borate (MESH:C109986),2-aminoethoxydiphenylborane (CHEBI:CHEBI:131184),False,True
3,PP2,Small Molecule,PPP2 (FPLX:PPP2),PP2 (CHEBI:CHEBI:78331),False,True
4,DCs,Cell types/Cell lines,DCX (HGNC:2714),Dendritic Cells (MESH:D003713),False,False



PMC:4644375: 
  Fixed VEGFR-3 -> Flt4 (UP)  [Gilda choice: FLT4 (HGNC)]
  Fixed Food -> food (CHEBI)  [Gilda choice: Food (MESH)]



,mention,type,Gilda pick,Model pick,Gilda Correct,Model Correct
0,VEGFR-3,Nonhuman Gene,FLT4 (HGNC:3767),Flt4 (UP:P35917),False,True
1,Vegfr2,Nonhuman Gene,KDR (HGNC:6307),Kdr (UP:P35918),False,False
2,Vegfr3,Nonhuman Gene,Flt4 (UP:P35917),Flt4 (UP:P35917),True,True
3,SMC,Cell types/Cell lines,DYM (HGNC:21317),DYM (HGNC:21317),False,False
4,fat,Tissue/Organ,CD36 (HGNC:1663),CD36 (HGNC:1663),False,False
5,Food,Small Molecule,Food (MESH:D005502),food (CHEBI:CHEBI:33290),False,True
6,feces,Tissue/Organ,Feces (MESH:D005243),Feces (MESH:D005243),True,True



PMC:4644379: 
  Fixed PP2 -> PP2 (CHEBI)  [Gilda choice: PPP2 (FPLX)]
  Fixed LAK -> Killer Cells, Lymphokine-Activated (MESH)  [Gilda choice: ALPK1 (HGNC)]



,mention,type,Gilda pick,Model pick,Gilda Correct,Model Correct
0,MICA,Human Gene,MICA (HGNC:7090),MICA (HGNC:7090),True,True
1,PP2,Small Molecule,PPP2 (FPLX:PPP2),PP2 (CHEBI:CHEBI:78331),False,True
2,LAK,Cell types/Cell lines,ALPK1 (HGNC:20917),"Killer Cells, Lymphokine-Activated (MESH:D015979)",False,True
